# 07 — Preparación de datos para el dashboard EPBI

> **Pipeline:** 01 Auditoría → 02 Limpieza ESS → 03 Eurostat → 04 Construcción EPBI → 05 Integración micro–macro → 06 Econometría → **07 Datos para dashboard**

Este notebook cierra el pipeline de Python. Su función no es volver a estimar modelos ni construir el dashboard visual, sino **transformar las salidas analíticas de los notebooks 05 y 06 en dos fuentes limpias y estables para Excel/VBA**.

La etapa combina:

- `epbi_micro_macro_panel.parquet`, generado en el notebook 05, para la parte descriptiva;
- `06_resultados_econometricos_final_limpio.xlsx`, generado en el notebook 06, para la parte econométrica.

El resultado es **un único libro Excel con exactamente dos hojas de datos**:

1. **`Microdatos_EPBI`** — individuos con EPBI válido y peso de análisis positivo, preparados para filtros, tablas dinámicas y gráficos descriptivos;
2. **`Econometria`** — coeficientes y estadísticas finales en formato largo, con etiquetas legibles para selectores y gráficos en VBA.

## Criterios de preparación

- Solo se exportan observaciones con EPBI válido y `analysis_weight > 0`.
- Las medias y porcentajes del EPBI deben calcularse en Excel/VBA utilizando `peso_analisis`.
- Los indicadores macro se repiten entre individuos del mismo contexto país-año y **no deben sumarse**.
- La hoja econométrica excluye constante, efectos fijos, bases individuales del spline y el modelo macro conjunto de diagnóstico.
- Las etiquetas técnicas de Patsy se traducen a nombres interpretables antes de exportar.
- La escritura del libro utiliza `pandas` y `openpyxl`, sin depender de `artifact_tool`.

El Excel generado será la capa de datos sobre la que se construye el dashboard final en VBA.


In [ ]:

from pathlib import Path
from datetime import datetime
import warnings
import math

import numpy as np
import pandas as pd

from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.worksheet.table import Table, TableStyleInfo
from openpyxl.utils import get_column_letter

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 220)

CURRENT_DIR = Path.cwd().resolve()
ROOT = CURRENT_DIR.parent if CURRENT_DIR.name.upper() == "CODIGO" else CURRENT_DIR

INPUT_MICRO = ROOT / "DATOS" / "PROCESADOS" / "epbi_micro_macro_panel.parquet"

# Salida habitual del Notebook 06
INPUT_ECON_PROJECT = ROOT / "RESULTADOS" / "TABLAS" / "06_resultados_econometricos_final_limpio.xlsx"

# Fallback útil si se ejecuta el notebook con el Excel en el mismo directorio
INPUT_ECON_LOCAL = CURRENT_DIR / "06_resultados_econometricos_final_limpio.xlsx"

OUTDIR = ROOT / "RESULTADOS" / "DASHBOARD"
OUTDIR.mkdir(parents=True, exist_ok=True)

OUTPUT = OUTDIR / "07_datos_dashboard_EPBI.xlsx"

print("Fecha:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("Microdatos:", INPUT_MICRO)
print("Resultados econométricos:", INPUT_ECON_PROJECT)
print("Salida:", OUTPUT)


## 1. Carga y selección de microdatos

La primera entrada procede del notebook 05. Se valida la estructura del panel micro–macro y se seleccionan únicamente los individuos con `valid_epbi == True`, EPBI no ausente y peso de análisis positivo, que son los casos utilizables en las visualizaciones ponderadas del dashboard.


In [ ]:

if not INPUT_MICRO.exists():
    raise FileNotFoundError(f"No se encontró el panel micro-macro: {INPUT_MICRO}")

micro = pd.read_parquet(INPUT_MICRO).copy()

required_micro = [
    "respondent_id", "cntry", "survey_year", "essround",
    "epbi", "epbi_abs", "valid_epbi", "analysis_weight",
    "gndr", "agea", "eisced", "mnactic",
    "lrscale", "gincdif", "trstplt", "stfdem",
    "gini", "riesgo_pobreza", "privacion_material_social_severa",
    "sobrecarga_vivienda", "renta_media_eur",
]

missing = [c for c in required_micro if c not in micro.columns]
if missing:
    raise KeyError("Faltan variables necesarias en el panel: " + ", ".join(missing))

# Solo observaciones válidas para EPBI y ponderación
mask = (
    micro["valid_epbi"].fillna(False)
    & micro["epbi"].notna()
    & micro["analysis_weight"].notna()
    & (pd.to_numeric(micro["analysis_weight"], errors="coerce") > 0)
)

micro = micro.loc[mask].copy()

print(f"Observaciones válidas EPBI + peso: {len(micro):,}")
print("Países:", micro["cntry"].nunique())
print("Años:", sorted(micro["survey_year"].dropna().unique().tolist()))


## 2. Etiquetas y variables orientadas al dashboard

Los microdatos analíticos conservan códigos técnicos útiles para el procesamiento, pero el dashboard necesita etiquetas comprensibles para filtros, títulos y leyendas. En esta sección se añaden nombres de país, categorías de sexo, educación, actividad y otras variables de presentación sin alterar las variables originales.


In [ ]:

# Nombres de países para facilitar mapas, títulos y filtros en Excel.
COUNTRY_NAMES = {
    "AL": "Albania", "AT": "Austria", "BE": "Bélgica", "BG": "Bulgaria",
    "CH": "Suiza", "CY": "Chipre", "CZ": "Chequia", "DE": "Alemania",
    "DK": "Dinamarca", "EE": "Estonia", "ES": "España", "FI": "Finlandia",
    "FR": "Francia", "GB": "Reino Unido", "GR": "Grecia", "HR": "Croacia",
    "HU": "Hungría", "IE": "Irlanda", "IL": "Israel", "IS": "Islandia",
    "IT": "Italia", "LT": "Lituania", "LU": "Luxemburgo", "LV": "Letonia",
    "ME": "Montenegro", "MK": "Macedonia del Norte", "NL": "Países Bajos",
    "NO": "Noruega", "PL": "Polonia", "PT": "Portugal", "RO": "Rumanía",
    "RS": "Serbia", "RU": "Rusia", "SE": "Suecia", "SI": "Eslovenia",
    "SK": "Eslovaquia", "TR": "Turquía", "UA": "Ucrania", "XK": "Kosovo"
}

SEX_LABELS = {1: "Hombre", 2: "Mujer"}

EISCED_LABELS = {
    0: "Menor que educación primaria",
    1: "Educación primaria",
    2: "Educación baja o primera etapa de secundaria",
    3: "Educación secundaria superior",
    4: "Educación post-secundaria no terciaria",
    5: "Educación terciaria de ciclo corto",
    6: "Grado universitario o equivalente",
    7: "Maestría, Doctorado o Educación Superior",
}

# Se aprovechan las etiquetas creadas en el Notebook 02 si existen.
if "sexo" not in micro.columns:
    micro["sexo"] = pd.to_numeric(micro["gndr"], errors="coerce").map(SEX_LABELS)

if "nivel_educativo" not in micro.columns:
    micro["nivel_educativo"] = pd.to_numeric(micro["eisced"], errors="coerce").map(EISCED_LABELS)

if "actividad_principal" not in micro.columns and "mnactic" in micro.columns:
    micro["actividad_principal"] = micro["mnactic"].astype("string")

if "age_group" not in micro.columns:
    bins = [15, 24, 34, 44, 54, 64, 74, np.inf]
    labels = ["16-24", "25-34", "35-44", "45-54", "55-64", "65-74", "75+"]
    micro["age_group"] = pd.cut(
        pd.to_numeric(micro["agea"], errors="coerce"),
        bins=bins,
        labels=labels,
        include_lowest=True
    )

micro["pais"] = micro["cntry"].astype(str).map(COUNTRY_NAMES).fillna(micro["cntry"].astype(str))


### 2.1 Columnas exportadas

Se selecciona el conjunto final de campos que necesita la capa VBA. Cuando resulta útil se conservan simultáneamente el código y su etiqueta legible.

La granularidad no cambia: **una fila sigue representando un individuo ESS**.


In [ ]:

micro_columns = [
    # Identificación / filtros
    ("respondent_id", "respondent_id"),
    ("cntry", "pais_codigo"),
    ("pais", "pais"),
    ("survey_year", "anio"),
    ("essround", "ronda"),

    # Perfil individual
    ("gndr", "sexo_codigo"),
    ("sexo", "sexo"),
    ("agea", "edad"),
    ("age_group", "grupo_edad"),
    ("eisced", "nivel_educativo_codigo"),
    ("nivel_educativo", "nivel_educativo"),
    ("mnactic", "actividad_codigo"),
    ("actividad_principal", "actividad_principal"),

    # Variables de la regresión
    ("lrscale", "ideologia_lr"),
    ("gincdif", "redistribucion_gincdif"),
    ("trstplt", "confianza_politicos"),
    ("stfdem", "satisfaccion_democracia"),

    # Construcción y resultado EPBI
    ("hinctnta", "decil_renta"),
    ("hincfel", "percepcion_ingresos_codigo"),
    ("objective_position", "posicion_objetiva"),
    ("subjective_position", "posicion_subjetiva"),
    ("epbi", "EPBI"),
    ("epbi_abs", "EPBI_abs"),
    ("epbi_category", "categoria_EPBI"),
    ("analysis_weight", "peso_analisis"),

    # Contexto macroeconómico utilizado en 06
    ("gini", "gini"),
    ("riesgo_pobreza", "riesgo_pobreza"),
    ("privacion_material_social_severa", "privacion_material_social_severa"),
    ("sobrecarga_vivienda", "sobrecarga_vivienda"),
    ("renta_media_eur", "renta_media_eur"),
]

# Solo se exportan columnas que realmente existan.
available_pairs = [(src, dst) for src, dst in micro_columns if src in micro.columns]

micro_dashboard = micro[[src for src, _ in available_pairs]].copy()
micro_dashboard.columns = [dst for _, dst in available_pairs]

# Convertir categorías/extensiones de pandas a tipos simples compatibles con Excel.
for c in micro_dashboard.columns:
    if str(micro_dashboard[c].dtype) in ("category", "string"):
        micro_dashboard[c] = micro_dashboard[c].astype(object)

micro_dashboard = micro_dashboard.sort_values(
    ["pais_codigo", "anio", "respondent_id"],
    kind="stable"
).reset_index(drop=True)

print("Dimensión Microdatos_EPBI:", micro_dashboard.shape)
display(micro_dashboard.head())


## 3. Lectura de los resultados econométricos

La segunda entrada procede del notebook 06. Se leen las hojas `Coeficientes_clave` y `Resumen_modelos`, que contienen los efectos sustantivos y las estadísticas necesarias para construir la vista econométrica del dashboard.


In [ ]:

if INPUT_ECON_PROJECT.exists():
    INPUT_ECON = INPUT_ECON_PROJECT
elif INPUT_ECON_LOCAL.exists():
    INPUT_ECON = INPUT_ECON_LOCAL
else:
    raise FileNotFoundError(
        "No se encontró 06_resultados_econometricos_final_limpio.xlsx "
        "ni en RESULTADOS/TABLAS ni en el directorio actual."
    )

print("Leyendo:", INPUT_ECON)

# Lectura estándar del Excel generado por el Notebook 06.
coef = pd.read_excel(INPUT_ECON, sheet_name="Coeficientes_clave")
summary = pd.read_excel(INPUT_ECON, sheet_name="Resumen_modelos")

print("Coeficientes:", coef.shape)
print("Resumen modelos:", summary.shape)
display(coef.head())


## 4. Construcción de la tabla econométrica para VBA

Los resultados se reorganizan en formato largo y se enriquecen con nombres de modelo, variable, categoría, referencia, dirección, significación e intervalos de confianza.

La sintaxis técnica de Patsy no se presenta al usuario final. En su lugar, cada efecto recibe una etiqueta interpretable, por ejemplo: `Grado universitario o equivalente (ref. Educación secundaria superior)`.


In [ ]:

# Nombres de modelos orientados al dashboard.
MODEL_LABELS = {
    "M1_comun": "M1 — Sociodemográfico",
    "M2_comun": "M2 — Individual completo",
    "M3a_gini": "M3a — Gini",
    "M3b_riesgo_pobreza": "M3b — Riesgo de pobreza",
    "M3c_privacion_material": "M3c — Privación material/social",
    "M3d_sobrecarga_vivienda": "M3d — Sobrecarga de vivienda",
    "M3e_renta_media": "M3e — Renta media",
    "M4a_educacion_x_gini": "M4a — Educación × Gini",
    "M4b_ideologia_x_gini": "M4b — Ideología × Gini",
}

MODEL_TYPE = {
    "M1_comun": "Principal",
    "M2_comun": "Principal",
    "M3a_gini": "Macro",
    "M3b_riesgo_pobreza": "Macro",
    "M3c_privacion_material": "Macro",
    "M3d_sobrecarga_vivienda": "Macro",
    "M3e_renta_media": "Macro",
    "M4a_educacion_x_gini": "Interacción exploratoria",
    "M4b_ideologia_x_gini": "Interacción exploratoria",
}

MODEL_ORDER = {
    "M1_comun": 1,
    "M2_comun": 2,
    "M3a_gini": 3,
    "M3b_riesgo_pobreza": 4,
    "M3c_privacion_material": 5,
    "M3d_sobrecarga_vivienda": 6,
    "M3e_renta_media": 7,
    "M4a_educacion_x_gini": 8,
    "M4b_ideologia_x_gini": 9,
}

# Etiquetas de las categorías utilizadas en la regresión.
SEX_ECON_LABELS = {
    2: "Mujer",
}

EDUCATION_ECON_LABELS = {
    0: "Menor que educación primaria",
    1: "Educación primaria",
    2: "Educación baja o primera etapa de secundaria",
    4: "Educación post-secundaria no terciaria",
    5: "Educación terciaria de ciclo corto",
    6: "Grado universitario o equivalente",
    7: "Maestría, Doctorado o Educación Superior",
}

ACTIVITY_ECON_LABELS = {
    2: "Educación / estudiante",
    3: "Desempleado, buscando trabajo",
    4: "Desempleado, no buscando trabajo",
    5: "Enfermedad o discapacidad permanente",
    6: "Jubilado",
    7: "Servicio comunitario o militar",
    8: "Tareas del hogar / cuidados",
    9: "Otra actividad",
}

REFERENCE_LABELS = {
    "sexo": "Hombre",
    "educacion": "Educación secundaria superior",
    "actividad": "Trabajo remunerado",
}

def extract_category_code(term):
    """Extrae el código contenido en [T.x] de un término Patsy."""
    import re
    m = re.search(r"\[T\.([0-9]+(?:\.0)?)\]", str(term))
    if not m:
        return None
    return int(float(m.group(1)))

def econometric_labels(term, variable_original):
    """
    Traduce un término técnico de Patsy a:
    variable_legible, categoria, referencia, tipo_efecto.
    """
    t = str(term)
    var = str(variable_original) if pd.notna(variable_original) else ""

    # Sexo
    if "C(gndr" in t:
        code = extract_category_code(t)
        categoria = SEX_ECON_LABELS.get(code, f"Categoría {code}")
        return "Sexo", categoria, REFERENCE_LABELS["sexo"], "Comparación categórica"

    # Educación (incluye interacción con Gini)
    if "C(eisced" in t:
        code = extract_category_code(t)
        categoria = EDUCATION_ECON_LABELS.get(code, f"Nivel educativo {code}")
        if ":gini" in t or "gini:" in t:
            return (
                "Educación × Gini",
                f"{categoria} × Gini",
                f"{REFERENCE_LABELS['educacion']} × Gini",
                "Interacción",
            )
        return "Nivel educativo", categoria, REFERENCE_LABELS["educacion"], "Comparación categórica"

    # Actividad principal
    if "C(mnactic" in t:
        code = extract_category_code(t)
        categoria = ACTIVITY_ECON_LABELS.get(code, f"Actividad {code}")
        return "Actividad principal", categoria, REFERENCE_LABELS["actividad"], "Comparación categórica"

    # Interacción ideología × Gini
    if ("lrscale:gini" in t) or ("gini:lrscale" in t):
        return "Ideología × Gini", "+1 punto de ideología × +1 punto de Gini", "—", "Interacción"

    # Variables continuas
    continuous = {
        "lrscale": ("Ideología izquierda-derecha", "+1 punto en la escala", "—"),
        "gincdif": ("Actitud redistributiva", "+1 punto en la escala", "—"),
        "trstplt": ("Confianza en políticos", "+1 punto en la escala", "—"),
        "stfdem": ("Satisfacción democrática", "+1 punto en la escala", "—"),
        "gini": ("Gini", "+1 punto", "—"),
        "riesgo_pobreza": ("Riesgo de pobreza", "+1 punto porcentual", "—"),
        "privacion_material_social_severa": (
            "Privación material/social severa", "+1 punto porcentual", "—"
        ),
        "sobrecarga_vivienda": ("Sobrecarga de vivienda", "+1 punto porcentual", "—"),
        "renta_media_miles_eur": ("Renta media", "+1.000 EUR", "—"),
    }

    if t in continuous:
        nombre, categoria, referencia = continuous[t]
        return nombre, categoria, referencia, "Efecto continuo"

    # Fallback: usar la variable ya legible del 06.
    return var if var else t, "—", "—", "Otro"


# El modelo macro conjunto fue diagnóstico y no se incorpora al dashboard.
coef = coef.loc[coef["modelo"].isin(MODEL_LABELS)].copy()

# El spline se mantiene como control, pero no se muestra en el forest plot.
coef = coef.loc[
    ~coef["termino"].astype(str).str.contains(r"bs\(agea", regex=True, na=False)
].copy()

# Asegurar tipos numéricos.
numeric_coef = [
    "coeficiente",
    "error_estandar_cluster_pais_ronda",
    "p_value_cluster_pais_ronda",
    "ci_95_inf",
    "ci_95_sup",
]
for c in numeric_coef:
    if c in coef.columns:
        coef[c] = pd.to_numeric(coef[c], errors="coerce")

for c in [
    "nobs", "paises", "rondas", "clusters_pais_ronda",
    "r_squared", "r_squared_adj", "aic", "bic"
]:
    if c in summary.columns:
        summary[c] = pd.to_numeric(summary[c], errors="coerce")

summary_keep = [
    c for c in [
        "modelo", "nobs", "paises", "rondas", "clusters_pais_ronda",
        "r_squared", "r_squared_adj", "aic", "bic"
    ] if c in summary.columns
]
summary_small = summary[summary_keep].drop_duplicates("modelo")

econometria = coef.merge(summary_small, on="modelo", how="left")

# Traducir cada término técnico a etiquetas humanas.
labels = econometria.apply(
    lambda r: econometric_labels(r["termino"], r.get("variable", "")),
    axis=1,
    result_type="expand",
)
labels.columns = ["variable_legible", "categoria", "referencia", "tipo_efecto"]

econometria = pd.concat([econometria, labels], axis=1)

econometria["orden_modelo"] = econometria["modelo"].map(MODEL_ORDER)
econometria["modelo_dashboard"] = econometria["modelo"].map(MODEL_LABELS)
econometria["tipo_modelo"] = econometria["modelo"].map(MODEL_TYPE)

econometria["significativo_5"] = np.where(
    econometria["p_value_cluster_pais_ronda"].notna()
    & (econometria["p_value_cluster_pais_ronda"] < 0.05),
    "Sí", "No"
)

econometria["direccion"] = np.select(
    [
        econometria["coeficiente"] > 0,
        econometria["coeficiente"] < 0,
    ],
    ["Positiva", "Negativa"],
    default="Nula"
)

econometria["inferencia"] = "Cluster país-ronda"
econometria["edad_control"] = "Spline cúbico (df=5)"

# Etiqueta corta idónea para gráficos.
econometria["etiqueta_grafico"] = np.where(
    econometria["referencia"].eq("—"),
    econometria["variable_legible"].astype(str) + " — " + econometria["categoria"].astype(str),
    econometria["categoria"].astype(str) + " (ref. " + econometria["referencia"].astype(str) + ")"
)

# IMPORTANTE:
# 'termino' ya NO se exporta. Solo se utiliza internamente para traducir Patsy.
econ_columns = [
    "orden_modelo",
    "modelo",
    "modelo_dashboard",
    "tipo_modelo",
    "variable_legible",
    "categoria",
    "referencia",
    "tipo_efecto",
    "etiqueta_grafico",
    "coeficiente",
    "error_estandar_cluster_pais_ronda",
    "p_value_cluster_pais_ronda",
    "ci_95_inf",
    "ci_95_sup",
    "significativo_5",
    "direccion",
    "nobs",
    "r_squared_adj",
    "paises",
    "rondas",
    "clusters_pais_ronda",
    "inferencia",
    "edad_control",
]

econometria = econometria[
    [c for c in econ_columns if c in econometria.columns]
].sort_values(
    ["orden_modelo", "variable_legible", "categoria"],
    kind="stable"
).reset_index(drop=True)

print("Dimensión Econometria:", econometria.shape)
display(
    econometria[
        [
            "modelo_dashboard", "variable_legible", "categoria",
            "referencia", "coeficiente", "p_value_cluster_pais_ronda"
        ]
    ].head(30)
)


## 5. Exportación a Excel

Las dos fuentes ya preparadas se escriben en un único libro con exactamente dos hojas:

- `Microdatos_EPBI`
- `Econometria`

Después de la escritura se aplican formatos básicos y se crean las tablas estructuradas que utilizará VBA. El objetivo es que el fichero pueda funcionar directamente como capa de datos del dashboard, sin pasos manuales intermedios.


In [ ]:

# Escritura inicial de las dos hojas.
# openpyxl es el motor estándar de Excel y evita depender de artifact_tool.
with pd.ExcelWriter(OUTPUT, engine="openpyxl") as writer:
    micro_dashboard.to_excel(
        writer,
        sheet_name="Microdatos_EPBI",
        index=False
    )
    econometria.to_excel(
        writer,
        sheet_name="Econometria",
        index=False
    )

# Formato posterior con openpyxl.
wb = load_workbook(OUTPUT)
ws_micro = wb["Microdatos_EPBI"]
ws_econ = wb["Econometria"]

header_fill = PatternFill("solid", fgColor="1F4E78")
header_font = Font(color="FFFFFF", bold=True)
header_alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

def format_header(ws):
    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = header_alignment
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

format_header(ws_micro)
format_header(ws_econ)

# Anchos controlados para no hacer autofit sobre más de 300.000 filas.
micro_widths = {
    "respondent_id": 20,
    "pais_codigo": 12,
    "pais": 20,
    "anio": 10,
    "ronda": 10,
    "sexo_codigo": 12,
    "sexo": 12,
    "edad": 10,
    "grupo_edad": 14,
    "nivel_educativo_codigo": 20,
    "nivel_educativo": 38,
    "actividad_codigo": 18,
    "actividad_principal": 30,
    "ideologia_lr": 15,
    "redistribucion_gincdif": 22,
    "confianza_politicos": 20,
    "satisfaccion_democracia": 24,
    "decil_renta": 14,
    "percepcion_ingresos_codigo": 26,
    "posicion_objetiva": 18,
    "posicion_subjetiva": 20,
    "EPBI": 12,
    "EPBI_abs": 12,
    "categoria_EPBI": 18,
    "peso_analisis": 16,
    "gini": 12,
    "riesgo_pobreza": 18,
    "privacion_material_social_severa": 32,
    "sobrecarga_vivienda": 22,
    "renta_media_eur": 18,
}

for idx, name in enumerate(micro_dashboard.columns, start=1):
    ws_micro.column_dimensions[get_column_letter(idx)].width = micro_widths.get(name, 16)

econ_widths = {
    "orden_modelo": 14,
    "modelo": 24,
    "modelo_dashboard": 34,
    "tipo_modelo": 24,
    "variable_legible": 34,
    "categoria": 42,
    "referencia": 38,
    "tipo_efecto": 24,
    "etiqueta_grafico": 55,
    "coeficiente": 14,
    "error_estandar_cluster_pais_ronda": 30,
    "p_value_cluster_pais_ronda": 26,
    "ci_95_inf": 14,
    "ci_95_sup": 14,
    "significativo_5": 16,
    "direccion": 14,
    "nobs": 14,
    "r_squared_adj": 16,
    "paises": 12,
    "rondas": 12,
    "clusters_pais_ronda": 22,
    "inferencia": 24,
    "edad_control": 28,
}

for idx, name in enumerate(econometria.columns, start=1):
    ws_econ.column_dimensions[get_column_letter(idx)].width = econ_widths.get(name, 18)

# Formatos numéricos.
micro_header = {c.value: c.column for c in ws_micro[1]}
for name in ["EPBI", "EPBI_abs", "posicion_objetiva", "posicion_subjetiva"]:
    if name in micro_header:
        col = get_column_letter(micro_header[name])
        for cell in ws_micro[col][1:]:
            cell.number_format = "0.000"

if "peso_analisis" in micro_header:
    col = get_column_letter(micro_header["peso_analisis"])
    for cell in ws_micro[col][1:]:
        cell.number_format = "0.0000"

for name in ["gini", "riesgo_pobreza", "privacion_material_social_severa", "sobrecarga_vivienda"]:
    if name in micro_header:
        col = get_column_letter(micro_header[name])
        for cell in ws_micro[col][1:]:
            cell.number_format = "0.00"

if "renta_media_eur" in micro_header:
    col = get_column_letter(micro_header["renta_media_eur"])
    for cell in ws_micro[col][1:]:
        cell.number_format = '#,##0.00'

econ_header = {c.value: c.column for c in ws_econ[1]}
for name in [
    "coeficiente", "error_estandar_cluster_pais_ronda",
    "p_value_cluster_pais_ronda", "ci_95_inf", "ci_95_sup",
    "r_squared_adj"
]:
    if name in econ_header:
        col = get_column_letter(econ_header[name])
        for cell in ws_econ[col][1:]:
            cell.number_format = "0.0000"

# Convertir los rangos en tablas estructuradas para VBA.
micro_ref = f"A1:{get_column_letter(ws_micro.max_column)}{ws_micro.max_row}"
econ_ref = f"A1:{get_column_letter(ws_econ.max_column)}{ws_econ.max_row}"

tbl_micro = Table(displayName="tblMicrodatosEPBI", ref=micro_ref)
tbl_micro.tableStyleInfo = TableStyleInfo(
    name="TableStyleMedium2",
    showFirstColumn=False,
    showLastColumn=False,
    showRowStripes=True,
    showColumnStripes=False,
)
ws_micro.add_table(tbl_micro)

tbl_econ = Table(displayName="tblEconometria", ref=econ_ref)
tbl_econ.tableStyleInfo = TableStyleInfo(
    name="TableStyleMedium2",
    showFirstColumn=False,
    showLastColumn=False,
    showRowStripes=True,
    showColumnStripes=False,
)
ws_econ.add_table(tbl_econ)

wb.save(OUTPUT)

print("\nExcel generado correctamente:")
print(OUTPUT)


## 6. Comprobaciones finales

Antes de cerrar el pipeline se valida que `Microdatos_EPBI` no contenga EPBI ausentes ni pesos no positivos y que `Econometria` no incluya el modelo macro conjunto ni términos técnicos que se han decidido excluir.

Estas comprobaciones garantizan que el libro entregado a VBA coincide con las reglas metodológicas fijadas en los notebooks anteriores.


In [ ]:

print("Microdatos_EPBI")
print("  Filas:", f"{len(micro_dashboard):,}")
print("  Columnas:", len(micro_dashboard.columns))
print("  EPBI missing:", micro_dashboard["EPBI"].isna().sum())
print(
    "  Peso no positivo:",
    (pd.to_numeric(micro_dashboard["peso_analisis"], errors="coerce") <= 0).sum()
)

print("\nEconometria")
print("  Filas:", len(econometria))
print("  Modelos:", econometria["modelo_dashboard"].drop_duplicates().tolist())
print(
    "  p-values missing:",
    econometria["p_value_cluster_pais_ronda"].isna().sum()
)

assert micro_dashboard["EPBI"].notna().all()
assert (
    pd.to_numeric(micro_dashboard["peso_analisis"], errors="coerce") > 0
).all()

assert "M3_conjunto_macro" not in econometria["modelo"].astype(str).unique()

# 'termino' solo se usa internamente para traducir Patsy y NO debe exportarse.
assert "termino" not in econometria.columns

# Comprobar que las etiquetas interpretables sí existen.
required_econ_labels = {
    "variable_legible",
    "categoria",
    "referencia",
    "tipo_efecto",
    "etiqueta_grafico",
}
missing_labels = required_econ_labels.difference(econometria.columns)
assert not missing_labels, f"Faltan columnas legibles: {missing_labels}"

print("\nValidación lógica superada.")
print("Archivo final:", OUTPUT)


## 7. Cierre del pipeline y notas para VBA

El libro generado contiene las dos tablas que alimentan el dashboard:

- fuente descriptiva: **`tblMicrodatosEPBI`**;
- fuente econométrica: **`tblEconometria`**.

Para utilizar correctamente estas fuentes:

- las medias y distribuciones del EPBI deben ponderarse con `peso_analisis`;
- los macrodatos no deben sumarse, porque se repiten entre individuos del mismo país-año;
- la tabla econométrica está preparada para seleccionar modelos y representar `coeficiente`, `ci_95_inf` y `ci_95_sup`;
- la edad se controla en la econometría mediante spline cúbico, pero sus bases se excluyen del forest plot porque no tienen una interpretación individual directa;
- para filtros y gráficos econométricos deben utilizarse las etiquetas legibles (`variable_legible`, `categoria`, `referencia`, `etiqueta_grafico`).

### Notas técnicas de esta versión

La implementación utiliza `pandas` + `openpyxl`, por lo que no depende de `artifact_tool`. Además, la columna técnica de Patsy se elimina antes de construir la tabla final y la validación se realiza sobre las columnas que realmente permanecen en `Econometria`, evitando comprobar campos eliminados deliberadamente.

Con esta exportación termina el procesamiento en Python. El siguiente paso ya no pertenece al pipeline analítico: consiste en consumir estas dos tablas desde **Excel/VBA** para construir la interfaz interactiva final.
